In [1]:
pip install --upgrade -r "requirements.txt" 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import http.client
import json
import bz2
import re
import requests

import pandas as pd

from collections import defaultdict
# from sentence_transformers import SentenceTransformer
# from keybert import KeyBERT
# topicexplorer import workset.tez
from gensim import corpora, models

import numpy as np

import string
from collections import Counter
from htrc_features import Volume
from htrc_features.resolvers import HttpResolver
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [3]:
# key_word_model = KeyBERT('distilbert-base-nli-mean-tokens')

Get stop words

In [4]:
stop_en = requests.get('https://data.analytics.hathitrust.org/data/text/default_stopwords_en.txt')
stop_en = stop_en.text
stop_en = stop_en.split()

Functions to get volume data

In [5]:
def retrieve_data(volume_id):
    # """Retrieve page-level JSON data for a volume."""

    # conn.request("GET", f"/ef-api/volumes/{volume_id}", headers=headers)
    # res = conn.getresponse()

    # if res.status != 200:
    #     print(f"API request failed: {res.status} {res.reason}")
    #     return "ERROR"

    # raw = res.read()

    res = requests.get("https://data.htrc.illinois.edu/ef-api/volumes/" + volume_id)
    raw = res.content

    if (res.status_code == 200):
        try:
            parsed = json.loads(raw.decode("utf-8"))
            return parsed
        except Exception as e:
            # print("Failed to parse JSON response")
            return "ERROR"
    else:
        return "NOT FOUND"


def tokens_count(record):
    ''' Create the list of words per page and how many time the word is mentioned '''

    word_key_count = Counter()
    
    for page in record['data']['features']['pages']:
        if page['body']:
            for token, pos_count in page['body']['tokenPosCount'].items():
                word_key_count[token] += sum(pos_count.values())
                    
    return word_key_count

def get_keywords(text):

    # extract tokens only
    dictionary = corpora.Dictionary([text.keys()])

    # Step 2: convert dict → BoW format
    bow = [(dictionary.token2id[word], count) for word, count in text.items()]

    # Step 3: corpus must be a list of documents
    corpus = [bow]

    # Step 4: run LDA
    lda_model = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=5)
    
    topics = ()
    for topic in lda_model.print_topics(num_topics=40, num_words=10):
        topics += topic

    return topics


In [6]:
def clean_df(data, stop_en):
    df = pd.DataFrame({'count': data})
    df = df.reset_index()
    df = df.rename(columns={'index': 'word'})
    # df = df.sort_values(by='count', ascending=False)
    df = df[~df['word'].isin(stop_en)]
    df = df[~df['word'].isin(stop.title() for stop in stop_en)]
    df = df[~df['word'].isin(list(string.punctuation))]
    df = df[~df['word'].isin(["'s","''","-RRB-","-LRB-","``","--","p."])]
    return df

Get workset volumes

In [26]:
import requests
workset = requests.get("https://worksets.hathitrust.org/ef-api/worksets/1abe5340-28d2-11f1-806f-2d821d8659ab")

workset_content = workset.content
# raw = json.loads(workset_content)

# print(workset_content)

In [27]:
workset_json=workset.json()
workset_json['data']['htids']

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [8]:
data = raw['gathers']
print(data)

readable_json = json.dumps(data, indent=4) # Indent with 4 spaces

x = readable_json.split()
y = "http://hdl.handle.net/2027/"
volumes = []
for i in x:
    if y in i:
        z = i.replace(y,"")
        z = z.replace('"',"")
        volumes.append(z)

[{'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h25m62k2k'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2sb3x99d'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2qv3cg1q'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2zc7s61z'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h27w67j0m'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h20c4sx21'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2nz8124z'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2j67981m'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2k64b51v'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2251fx5r'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2cf9jj9k'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h20863h80'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h24x54v0f'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2rr1q00m'}, {'id': 'http://hdl.handle.net/2027/ucbk.ark:/28722/h2z892s80'}, {'id': 'http://hdl.handle.net/2027/ucbk

Handle Volume

In [17]:
worked = 0
failed = 0
noget = 0
for vol in volumes:
    # print("https://data.htrc.illinois.edu/ef-api/volumes/" + vol)
    # res = requests.get("https://data.htrc.illinois.edu/ef-api/volumes/" + vol)
    # raw = res.content

    # if (res.status_code == 200):
    #     try:
    #         parsed = json.loads(raw.decode("utf-8"))
    #         print('worked')
    #         worked += 1
    #     except Exception as e:
    #         failed += 1
    #         print(e)
    # else: 
    #     noget += 1

    data = retrieve_data(vol)
    print(data)

    if (data == "NOT FOUND"): 
        notget += 1
    elif (data == "ERROR"):
        failed += 1
    else: 
        worked += 1

print ("WORKED: " + str(worked))
print ("FAILED: " + str(failed))
print ("NOT GOT: " + str(failed))

NOT FOUND


NameError: name 'notget' is not defined

Testing with just two working volumes

In [10]:
parsed1 = retrieve_data('uiug.30112104141319')
parsed2 = retrieve_data('uiug.30112104141103')
print(parsed1)
print(parsed2)

{'code': 200, 'data': {'@context': 'https://worksets.htrc.illinois.edu/context/ef_context.jsonld', 'schemaVersion': 'https://schemas.hathitrust.org/EF_Schema_v_3.0', 'id': 'https://data.analytics.hathitrust.org/extracted-features/20200210/uiug.30112104141319', 'htid': 'uiug.30112104141319', 'type': 'DataFeed', 'publisher': {'id': 'https://analytics.hathitrust.org', 'type': 'Organization', 'name': 'HathiTrust Research Center'}, 'datePublished': 20200210, 'metadata': {'schemaVersion': 'https://schemas.hathitrust.org/EF_Schema_MetadataSubSchema_v_3.0', 'id': 'http://hdl.handle.net/2027/uiug.30112104141319', 'type': ['DataFeedItem', 'PublicationVolume'], 'dateCreated': 20200209, 'title': 'Corpsman.', 'contributor': {'id': 'http://www.viaf.org/viaf/152739208', 'type': 'http://id.loc.gov/ontologies/bibframe/Organization', 'name': 'Job Corps (U.S.)'}, 'pubDate': 1967, 'publisher': {'id': 'http://catalogdata.library.illinois.edu/lod/entities/ProvisionActivityAgent/ht/Job%20Corps', 'type': 'htt

In [11]:
counts1 = tokens_count(parsed1)
counts2 = tokens_count(parsed2)
print(counts1)
print(counts2)

Counter({'.': 410, 'the': 297, ',': 214, 'to': 174, 'a': 163, 'and': 160, 'in': 133, 'of': 133, 'is': 66, 'The': 63, "''": 63, '``': 61, 'I': 57, "'s": 56, 'that': 53, 'was': 50, 'for': 48, '-LRB-': 48, '-RRB-': 48, 'they': 46, 'Corpsmen': 39, 'are': 38, 'Job': 38, 'Corps': 38, 'it': 37, 'with': 35, 'be': 35, 'at': 34, 'you': 34, 'he': 33, 'as': 31, 'on': 31, 'by': 29, 'their': 28, 'have': 27, 'center': 26, 'Corpsman': 25, 'were': 25, 'out': 24, 'from': 24, 'do': 24, 'all': 23, 'an': 23, "n't": 22, 'this': 22, 'It': 20, 'not': 19, 'like': 19, ':': 19, 'we': 19, 'would': 18, 'up': 18, '?': 18, 'his': 17, '-': 17, 'time': 17, 'about': 17, 'one': 17, 'she': 17, 'They': 16, 'But': 16, 'should': 16, 'people': 15, 'In': 15, 'had': 15, 'two': 15, 'He': 15, 'other': 15, 'but': 14, 'been': 14, 'who': 14, 'them': 14, 'good': 14, 'just': 13, 'over': 13, 'when': 13, 'work': 13, 'THE': 12, 'has': 12, 'into': 12, 'Larry': 12, 'years': 11, 'help': 11, 'through': 11, 'because': 11, 'think': 11, 'get':

In [12]:
df1 = clean_df(counts1, stop_en)
df1.nlargest(10, 'count')

,word,count
170,Corpsmen,39
222,Job,38
449,Corps,38
102,center,26
93,Corpsman,25
448,n't,22
411,time,17
24,people,15
614,good,14
727,work,13


In [13]:
df2 = clean_df(counts2, stop_en)
df2.nlargest(10, 'count')

,word,count
98,Corpsmen,56
320,Corps,55
140,Job,48
66,center,36
585,n't,36
60,Corpsman,29
19,people,21
2219,Negro,21
65,THE,19
46,Washington,18


In [14]:
dict1 = df1.set_index('word')['count'].to_dict()
print(get_keywords(dict1))

(0, '0.009*"Corps" + 0.007*"Job" + 0.007*"Corpsmen" + 0.005*"center" + 0.004*"Corpsman" + 0.004*"n\'t" + 0.003*"time" + 0.003*"THE" + 0.003*"years" + 0.002*"Newman"', 1, '0.007*"Corpsman" + 0.007*"Corps" + 0.006*"Corpsmen" + 0.006*"Job" + 0.005*"center" + 0.004*"n\'t" + 0.004*"good" + 0.003*"people" + 0.003*"work" + 0.003*"time"', 2, '0.011*"Corpsmen" + 0.009*"Job" + 0.007*"center" + 0.007*"Corps" + 0.006*"n\'t" + 0.005*"Corpsman" + 0.004*"good" + 0.003*"work" + 0.003*"people" + 0.003*"time"', 3, '0.008*"Corpsmen" + 0.007*"Job" + 0.006*"Corps" + 0.004*"n\'t" + 0.004*"Corpsman" + 0.004*"time" + 0.004*"center" + 0.003*"people" + 0.003*"Larry" + 0.003*"good"', 4, '0.010*"Corps" + 0.009*"Job" + 0.008*"Corpsmen" + 0.006*"center" + 0.005*"Corpsman" + 0.005*"n\'t" + 0.004*"time" + 0.004*"people" + 0.003*"work" + 0.003*"Larry"')


In [15]:
dict2 = df2.set_index('word')['count'].to_dict()
print(get_keywords(dict2))

(0, '0.007*"Corps" + 0.007*"Corpsmen" + 0.006*"Job" + 0.005*"center" + 0.004*"Corpsman" + 0.004*"n\'t" + 0.004*"Negro" + 0.003*"man" + 0.003*"people" + 0.003*"THE"', 1, '0.008*"Corpsmen" + 0.007*"Corps" + 0.006*"n\'t" + 0.006*"Job" + 0.005*"Corpsman" + 0.005*"center" + 0.003*"Negro" + 0.003*"people" + 0.003*"man" + 0.003*"back"', 2, '0.009*"Corps" + 0.007*"Corpsmen" + 0.007*"Job" + 0.005*"center" + 0.005*"n\'t" + 0.004*"Corpsman" + 0.003*"program" + 0.003*"world" + 0.003*"home" + 0.003*"photo"', 3, '0.009*"Corpsmen" + 0.008*"Corps" + 0.007*"Job" + 0.006*"n\'t" + 0.005*"center" + 0.004*"Corpsman" + 0.003*"Negro" + 0.003*"people" + 0.003*"THE" + 0.003*"Washington"', 4, '0.009*"Corpsmen" + 0.008*"Job" + 0.007*"Corps" + 0.005*"center" + 0.004*"n\'t" + 0.004*"Corpsman" + 0.003*"people" + 0.003*"THE" + 0.003*"photo" + 0.003*"great"')


In [ ]:
# = for \
# + for :

print(retrieve_data('ucbk.ark+=28722=h2ws8hz0x'))

{'code': 404, 'message': 'Volume ucbk.ark:/28722/h2ws8hz0x not found'}
